In [1]:
import pandas as pd
import mysql.connector
import ollama

In [4]:
connection = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "jessepinkman",
    database = "company_db",
    auth_plugin="mysql_native_password"
)

In [5]:
cursor = connection.cursor(buffered=True)

In [6]:
#we will give our database schema to the LLM so he can understand our database and the tables inside it
schema = """
Table: employees
Columns: employee_id, employee_name, age, gender, department_id, salary, joining_date, city

Table: departments
Columns: department_id, department_name

Table: customers
Columns: customer_id, customer_name, gender, age, city, phone_number, email, registration_date

Table: products
Columns: product_id, product_name, category, brand, price, stock_quantity

Table: orders
Columns: order_id, customer_id, product_id, employee_id, quantity, order_amount, order_date, payment_method, order_status

Table: sales
Columns: sale_id, employee_id, product_name, sale_amount, sale_date, region
"""

In [41]:
user_question = "count total employees in the company"

In [42]:
#Creating a special prompt so LLM can understand properly what task to do we gonna use ollama as an llm model, cause we dont have money to buy openai authentication key.
prompt = f"""
You are an SQL query generator.

Your task is to convert the user question into a valid MySQL query.

Database Schema:

{schema}

Rules:
1. Return only SQL query
2. Do not explain anything
3. Do not return English sentence
4. Use only the tables and columns provided
5. SQL must be valid MySQL syntax

User Question:
{user_question}
"""

In [43]:
response = ollama.chat(
    model = 'llama3',
    messages = [
        {
            "role":"user",
            "content":prompt
        }
    ]
)

In [44]:
generated_sql = response.message.content
print(generated_sql)

SELECT COUNT(*) AS total_employees 
FROM employees;


In [46]:
cursor.execute(generated_sql)

result = cursor.fetchall()

columns = [desc[0] for desc in cursor.description]

query_result = pd.DataFrame(result, columns=columns)

query_result

,total_employees
0,30
